# LivePortrait Colab Backup for OpenShorts

Gunakan notebook ini sebagai cadangan (fallback) jika Hugging Face Zero-GPU sedang penuh atau lambat. Notebook ini akan menjalankan LivePortrait di GPU Colab (T4 gratis) dan mengeksposnya via Localtunnel agar bisa diakses oleh OpenShorts di lokal Anda.

In [ ]:
!pip install -q fastapi uvicorn python-multipart moviepy gradio_client
!npm install -g localtunnel
!git clone https://github.com/KlingTeam/LivePortrait.git || true
!cd LivePortrait && pip install -r requirements.txt -q

In [ ]:
import os
import threading
import subprocess
import time
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import FileResponse
import uvicorn

# Setup FastAPI
app = FastAPI()
os.makedirs("/content/outputs", exist_ok=True)

@app.get("/")
def read_root():
    return {"status": "LivePortrait Backend Running"}

@app.post("/predict")
async def predict(image: UploadFile = File(...), audio: UploadFile = File(...)):
    image_path = f"/content/{image.filename}"
    audio_path = f"/content/{audio.filename}"
    
    with open(image_path, "wb") as f:
        f.write(await image.read())
    with open(audio_path, "wb") as f:
        f.write(await audio.read())
        
    # Note: Running actual LivePortrait locally requires downloading weights.
    # For simplicity, we can use the Gradio Client if this Colab is just a bridge,
    # OR run the full inference script locally. Here we run the local inference script:
    output_name = f"out_{int(time.time())}.mp4"
    output_path = f"/content/outputs/{output_name}"
    
    print(f"Generating portrait... this will take a moment.")
    
    cmd = [
        "python", "/content/LivePortrait/inference.py",
        "--source", image_path,
        "--driving_audio", audio_path,
        "--output_dir", "/content/outputs",
        "--flag_do_crop"
    ]
    
    try:
        subprocess.run(cmd, check=True)
        # LivePortrait saves to a specific format inside output_dir
        # usually <source_name>--<driving_name>.mp4
        src_name = os.path.splitext(os.path.basename(image_path))[0]
        drv_name = os.path.splitext(os.path.basename(audio_path))[0]
        expected_out = f"/content/outputs/{src_name}--{drv_name}.mp4"
        
        if os.path.exists(expected_out):
            return FileResponse(expected_out, media_type="video/mp4", filename="output.mp4")
        else:
            # Fallback to returning the latest video in the folder
            vids = [f for f in os.listdir("/content/outputs") if f.endswith('.mp4')]
            if vids:
                latest = max(vids, key=lambda v: os.path.getctime(f"/content/outputs/{v}"))
                return FileResponse(f"/content/outputs/{latest}", media_type="video/mp4", filename="output.mp4")
            return {"error": "Generation failed, no output found."}
    except Exception as e:
        return {"error": str(e)}

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

# Start server in background thread
thread = threading.Thread(target=run_server, daemon=True)
thread.start()

import time
time.sleep(3) # Wait for server to start


In [ ]:
import urllib
print("===============================================================")
print("DAPATKAN PASSWORD INI DAN MASUKKAN DI WEBSITE LOCALTUNNEL:")
password = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n")
print("Password IP Anda:", password)
print("===============================================================")

# Start localtunnel
!lt --port 8000

### Cara Penggunaan:
1. Klik tombol **Run All** (Runtime -> Run all) di Colab.
2. Copy **Password IP Anda** yang muncul di atas.
3. Klik link Localtunnel (`https://xxx.loca.lt`) yang muncul di output terakhir.
4. Paste password tersebut di halaman Localtunnel lalu klik "Click to Continue".
5. Copy URL Localtunnel Anda (`https://xxx.loca.lt`) dan masukkan di kolom **Colab Backup URL** pada Settings di OpenShorts lokal Anda.